# Cluster Analysis: Poetic Features

In [2]:
import warnings

warnings.filterwarnings("ignore")

from mqdq import babble, elegy

import numpy as np
import pandas as pd

import umap

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from typing import Callable, Collection, Union

import glob

In [3]:
from pathlib import Path


# Project paths

def find_metrical_root():
    current = Path.cwd().resolve()

    for folder in [current, *current.parents]:
        if (folder / "corpus").is_dir():
            return folder

    raise FileNotFoundError("Metrical analysis root not found.")


BASE_DIR = find_metrical_root()
CORPUS_DIR = BASE_DIR / "corpus"


# Find XML files

files = sorted(CORPUS_DIR.glob("*.xml"))

files = [
    fn for fn in files
    if fn.name != "X_operibus.xml"
]


# Build collection

collection = []

for fn in files:
    parts = fn.stem.split("_")

    author = parts[0]
    title = parts[1]

    print("Sto leggendo:", fn.name)

    b = babble.Babbler.from_file(fn, name=title)
    b.author = author

    collection.append(b)

Sto leggendo: AE_bhc.xml
Lines: 700 Corrupt: 32 Total Dropped: 62
Sto leggendo: ASt_troilus.xml
Lines: 6114 Corrupt: 154 Total Dropped: 306
Sto leggendo: BB_adelae.xml
Lines: 1366 Corrupt: 31 Total Dropped: 62
Sto leggendo: BBi_speculum.xml
Lines: 1954 Corrupt: 47 Total Dropped: 92
Sto leggendo: BI_reynardus.xml
Lines: 1782 Corrupt: 86 Total Dropped: 164
Sto leggendo: BM_regum.xml
Lines: 914 Corrupt: 0 Total Dropped: 0
Sto leggendo: BS_mathematicus.xml
Lines: 854 Corrupt: 7 Total Dropped: 14
Sto leggendo: EM_mahumeti.xml
Lines: 1148 Corrupt: 18 Total Dropped: 36
Sto leggendo: HA_pentateuchum.xml
Lines: 1010 Corrupt: 34 Total Dropped: 66
Sto leggendo: HM_gestis.xml
Lines: 1390 Corrupt: 41 Total Dropped: 78
Sto leggendo: HW_hortus.xml
Lines: 9068 Corrupt: 0 Total Dropped: 0
Sto leggendo: H_carmina.xml
Lines: 564 Corrupt: 2 Total Dropped: 6
Sto leggendo: H_mysterio.xml
Lines: 606 Corrupt: 0 Total Dropped: 0
Sto leggendo: IS_entheticus.xml
Lines: 1852 Corrupt: 17 Total Dropped: 34
Sto legg

In [4]:
for b in collection:
    print(f"Author: {b.author}, Name: {b.name}")

Author: AE, Name: bhc
Author: ASt, Name: troilus
Author: BB, Name: adelae
Author: BBi, Name: speculum
Author: BI, Name: reynardus
Author: BM, Name: regum
Author: BS, Name: mathematicus
Author: EM, Name: mahumeti
Author: HA, Name: pentateuchum
Author: HM, Name: gestis
Author: HW, Name: hortus
Author: H, Name: carmina
Author: H, Name: mysterio
Author: IS, Name: entheticus
Author: M, Name: ars
Author: M, Name: epistulae
Author: M, Name: tobias
Author: NLC, Name: miracula
Author: NLC, Name: speculum
Author: PDE, Name: rebussiculis
Author: PR, Name: aurora
Author: QS, Name: alexandri
Author: RDV, Name: paulino
Author: RL, Name: anselmi
Author: RT, Name: epistulae
Author: RT, Name: memorabilibus
Author: RT, Name: miracula
Author: SAC, Name: excidio
Author: SR, Name: draco
Author: U1, Name: pyramo
Author: U2, Name: ysengrimus
Author: U3, Name: guiardinus
Author: U4, Name: liberregum
Author: X, Name: cilr


In [5]:
sz = 200  
collection_chunked = []  

for b in collection:
    ll = b.raw_source
    chunks = [ll[i : i + sz] for i in range(0, len(ll), sz)]

    for i, chunk in enumerate(chunks):
        if len(chunk) == sz:  
            new_babbler = babble.Babbler(
                chunk, name=f"{b.name} - {i+1}"
            )  
            new_babbler.author = b.author  
            collection_chunked.append(new_babbler)

print([f"{x.author} - {x.name} - {len(x)}" for x in collection_chunked])

['AE - bhc - 1 - 200', 'AE - bhc - 2 - 200', 'AE - bhc - 3 - 200', 'ASt - troilus - 1 - 200', 'ASt - troilus - 2 - 200', 'ASt - troilus - 3 - 200', 'ASt - troilus - 4 - 200', 'ASt - troilus - 5 - 200', 'ASt - troilus - 6 - 200', 'ASt - troilus - 7 - 200', 'ASt - troilus - 8 - 200', 'ASt - troilus - 9 - 200', 'ASt - troilus - 10 - 200', 'ASt - troilus - 11 - 200', 'ASt - troilus - 12 - 200', 'ASt - troilus - 13 - 200', 'ASt - troilus - 14 - 200', 'ASt - troilus - 15 - 200', 'ASt - troilus - 16 - 200', 'ASt - troilus - 17 - 200', 'ASt - troilus - 18 - 200', 'ASt - troilus - 19 - 200', 'ASt - troilus - 20 - 200', 'ASt - troilus - 21 - 200', 'ASt - troilus - 22 - 200', 'ASt - troilus - 23 - 200', 'ASt - troilus - 24 - 200', 'ASt - troilus - 25 - 200', 'ASt - troilus - 26 - 200', 'ASt - troilus - 27 - 200', 'ASt - troilus - 28 - 200', 'ASt - troilus - 29 - 200', 'BB - adelae - 1 - 200', 'BB - adelae - 2 - 200', 'BB - adelae - 3 - 200', 'BB - adelae - 4 - 200', 'BB - adelae - 5 - 200', 'BB -

In [6]:
collection_vect = elegy.vectorise_babs(collection_chunked)

In [7]:
collection_vect.to_csv("output.csv", index=False, encoding="utf-8")